# fhir4ds.fhirpath

High-performance FHIRPath R4 evaluation. This notebook demonstrates how to evaluate FHIRPath expressions both using the standalone Python evaluator and the ultra-fast DuckDB C++ backend.

In [ ]:
# Install the unified package
%pip install fhir4ds-v2

## 1. Standalone Python Evaluator
Ideal for processing individual resources in application logic.

In [ ]:
from fhir4ds.fhirpath import evaluate

patient = {
    "resourceType": "Patient",
    "active": True,
    "name": [
        {"use": "official", "family": "Smith", "given": ["Jane", "Marie"]},
        {"use": "nickname", "given": ["Janie"]},
    ],
    "gender": "female",
    "birthDate": "1985-03-15",
    "telecom": [
        {"system": "phone", "value": "555-1234", "use": "home"},
        {"system": "email", "value": "jane@example.com"},
    ]
}

# Get all given names
evaluate(patient, "Patient.name.given")

## 2. SQL-Native High-Performance Evaluation
For population-scale analytics, FHIR4DS registers FHIRPath as a native DuckDB function. This uses the high-performance C++ backend.

In [ ]:
import fhir4ds
import json

con = fhir4ds.create_connection()

# Sample data in a SQL table
con.execute("CREATE TABLE resources (resource JSON)")
con.execute("INSERT INTO resources VALUES (?)", [json.dumps(patient)])

# Query using native fhirpath() function
con.execute("""
    SELECT 
        fhirpath_text(resource, 'Patient.name.family.first()') as family,
        fhirpath(resource, 'Patient.name.given') as all_given
    FROM resources
""").df()

### Complex Filtering in SQL
You can use FHIRPath within `WHERE` clauses for precise cohort selection.

In [ ]:
con.execute("""
    SELECT count(*)
    FROM resources
    WHERE fhirpath_bool(resource, 'Patient.birthDate < @1990-01-01')
""").fetchone()